In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

import warnings
warnings.filterwarnings("ignore")

# 第 4 课：3D (4D) Vision 与神经场实战

本章节将带领大家从经典的相机几何学出发，逐步过渡到现代的神经表示技术，揭开 3D 重建与渲染的神秘面纱。

## 引言与背景
计算机视觉的研究最早可以追溯到如何让机器理解 3D 世界（“What” and “Where”）。与大家在前面的课程（比如分类、检测任务）中接触的 2D CNN 处理像素特征不同，**3D 视觉聚焦于场景的空间几何架构与物理光照属性**。

### 3D 信息的常见表征形式
传统视觉与几何处理领域通常使用显式表达：
1. **点云 (Point Cloud):** 数据通过三维坐标集合表示，获取直观。缺点是离散、缺乏连接性。
2. **体素 (Voxel):** 类似于 2D 像素的 3D 扩展方式。缺点是极其消耗内存（空间分辨率受限）。
3. **网格 (Mesh):** 由顶点和面片组成，是渲染引擎（如 Unreal、Unity）最常用的标准形式。

现代 3D 视觉越来越倾向于**神经隐式场 (Neural Implicit Fields)** 的表达：用神经网络表示连续的三维空间，如 **NeRF**（神经辐射场）。它可以突破分辨率限制，在内存占用小的情况下实现高度逼真的新视角合成。

---

## 1. 相机几何与坐标变换 (下钻)

三维世界的点是如何投影到一张 2D 图片上的？

### 1.1 针孔相机模型与内参分解
针孔相机模型是 3D 投影的基础。3D 点 $(X, Y, Z)$ 投影到图像平面 $(u, v)$ 的方程由**内参矩阵 $K$** 决定：

$$K = \begin{bmatrix} f_x & s & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$

- **$f_x, f_y$**: 像素焦距，定义了图像的尺度。
- **$c_x, c_y$**: 主点坐标，通常是图像的几何中心。
- **$s$**: 倾斜因子 (Skew)，描述像素单元格的倾斜程度（现代相机通常为 0）。

### 1.2 坐标转换链
从 3D 世界到位图的精确计算链条（极其关键）：
1. **世界坐标 ($P_w$) → 相机坐标 ($P_c$):** 
   $$P_c = R P_w + t$$
   其中 $R (3\times3), t(3\times1)$ 构成了相机的**外参**。
2. **相机坐标 ($P_c$) → 归一化平面:** 
   $$[x', y', 1]^T = [X_c/Z_c, Y_c/Z_c, 1]^T$$
3. **归一化平面 → 像素坐标 ($u, v$):** 
   $$\begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = K \begin{bmatrix} x' \\ y' \\ 1 \end{bmatrix}$$

### 1.3 可微的 PyTorch 投影仪演示
深度学习需要反向传播来优化网络，因此我们需要几何操作是**可微的** (Differentiable)。这里我们纯用 PyTorch 模拟 3D 点通过 $R, t, K$ 投影到图片上的过程。

In [ ]:
class DifferentiableProjector(nn.Module):
    """
    使用 PyTorch 实现一个可微的投影层
    """
    def __init__(self, fx, fy, cx, cy):
        super().__init__()
        # 把内参注册为不需要梯度的 buffer
        self.register_buffer('K', torch.tensor([
            [fx, 0, cx],
            [0, fy, cy],
            [0, 0, 1]
        ]).float())

    def forward(self, points_3d, R, t):
        """
        points_3d: (N, 3) 空间中的点
        R: (3, 3) 旋转矩阵
        t: (3,) 平移矩阵
        """
        # 1. 外参变换 (从世界坐标系转到相机坐标系)
        # points_cam: (N, 3)
        points_cam = (torch.matmul(R, points_3d.t()) + t.view(3, 1)).t()
        
        # 2. 透视除法 (归一化平面: x/z, y/z)
        z = points_cam[:, 2:] + 1e-6 # 加个极小值避免除零
        points_norm = points_cam[:, :2] / z
        
        # 3. 内参投影 (变换到像素坐标 u, v)
        u = self.K[0, 0] * points_norm[:, 0] + self.K[0, 2]
        v = self.K[1, 1] * points_norm[:, 1] + self.K[1, 2]
        
        # 组合成输出形状 (N, 2)
        return torch.stack([u, v], dim=1)

# ============== 演示可微性 (Gradient Flow) ==============
points = torch.randn(10, 3, requires_grad=True) # 初始化10个空间点，允许算梯度
R = torch.eye(3)
t = torch.zeros(3)
projector = DifferentiableProjector(fx=500, fy=500, cx=320, cy=240)

uv = projector(points, R, t)
loss = uv.sum()  # 随便算个loss
loss.backward()  # 反向传播！
print("梯度流动成功! 这是第一个特征点的梯度:\n", points.grad[0])

---

## 2. 传统 3D 重建与几何约束

### 2.1 对极几何 (Epipolar Geometry)
两台相机观察同一个 3D 点时，存在约束关系。由**本质矩阵 $E$** 刻画：
$$x_2^T E x_1 = 0, \quad E = [t]_{\times} R$$
其中 $[t]_{\times}$ 是平移向量的反对称矩阵。对于未标定相机，使用**基础矩阵 $F$**: $F = K_2^{-T} E K_1^{-1}$。

### 2.2 SfM (Structure from Motion) 流程
从一组图像中恢复 3D 结构和相机位姿的典型流程：
1. **特征提取与匹配**: 如 SIFT, ORB 或深度学习特征 (SuperPoint)。
2. **特征匹配**: 构建多视图之间的对应关系。
3. **对极几何计算**: 恢复相对 R, t。
4. **三角测量 (Triangulation)**: 根据对应点计算 3D 空间坐标。
5. **BA 优化 (Bundle Adjustment)**: 同时优化相机位姿和 3D 点坐标，最小化重投影误差。
    $$\min_{P, X} \sum_{i,j} \|x_{i,j} - \text{project}(P_i, X_j)\|^2$$

---

## 3. 神经辐射场 (NeRF) 深度探索实战

**NeRF (Neural Radiance Fields)** 是 2020 年 ECCV Best Paper，彻底颠覆了新视角合成领域。它摒弃了显式的 Voxel 或 Mesh，转而用神经网络记住整个场景。

### 3.1 核心原理: MLP 作为场景函数
NeRF 利用 MLP 学习一个连续的场景函数 $f(x, y, z, \theta, \phi) \to (r,g,b, \sigma)$：
- 输入：空间三维坐标 $(x, y, z)$ 加上 观察视角 $(\theta, \phi)$（单位方向向量 $d$）
- 输出：该点发出的颜色 $(r, g, b)$，以及体密度 $\sigma$（不透明度）。

---
### 3.2 步骤一：位置编码 (Positional Encoding)

**现象**：神经网络天生倾向于学习低频函数（平滑区域）。如果直接向 MLP 输入 $(x, y, z)$，渲染出来的图片会非常模糊。
**原理**：使用 $\sin$ 和 $\cos$ 原语将低维的坐标映射到高维，这和傅里叶变换类似，能够促使网络保留并学习到高频的边缘纹理。

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, L=10):
        super().__init__()
        # 定义需要编码的频率阶数 L
        self.L = L
        # 计算一系列频率的底数 2^0, 2^1, ..., 2^{L-1}
        # 这里的频率倍增是核心
        self.freq_bands = 2.0 ** torch.linspace(0, L - 1, steps=L)
        
    def forward(self, x):
        """
        x 形状为 (N, 3)，N 是采样点数量
        """
        encoded = [x] # 原始坐标也保留
        
        # 对于所有预设好的频率，分别求 sin 和 cos
        for freq in self.freq_bands:
            encoded.append(torch.sin(x * freq * torch.pi))
            encoded.append(torch.cos(x * freq * torch.pi))
            
        # 沿着特征维度进行拼接
        return torch.cat(encoded, dim=-1)

# 测试一下 Positional Encoding
pe = PositionalEncoding(L=10) # 10阶
sample_points = torch.randn(5, 3) # 5个三维点
out_features = pe(sample_points)

print(f"原始特征维度: {sample_points.shape}")
# 输出结果应当是 3 + 3*2*10 = 63 维
print(f"编码后特征维度: {out_features.shape}") 

---
### 3.3 步骤二：光栅化采样 vs 体渲染积分

要合成一张图像，NeRF 采用的不是像 Mesh 一样的光栅化方法，而是模拟物理学的**光线追踪体渲染**。沿着一根从相机原点射出的射线采样 $N$ 个点：

$$Ray(t) = O + tD$$
其中 $O$ 是相机原点坐标，$D$ 是射线的方向向量，$t$ 是距离深度。

In [ ]:
def compute_ray_samples(ray_origin, ray_dir, z_near=2.0, z_far=6.0, num_samples=64):
    """
    在射线上均匀进行采样
    """
    N_rays = ray_origin.shape[0]
    
    # 在 near 和 far 之间生成均匀深度的采样分层 t_vals
    t_vals = torch.linspace(z_near, z_far, num_samples).expand(N_rays, num_samples)
    
    # Ray(t) = Origin + t * Direction
    # ray_origin: (N_rays, 3) -> (N_rays, 1, 3)
    # ray_dir: (N_rays, 3) -> (N_rays, 1, 3)
    # t_vals: (N_rays, num_samples) -> (N_rays, num_samples, 1)
    
    pts = ray_origin.unsqueeze(1) + ray_dir.unsqueeze(1) * t_vals.unsqueeze(-1)
    
    return pts, t_vals

# 测试一次射线采样
O = torch.tensor([[0.0, 0.0, 0.0]]) # 假设相机在世界中心
D = torch.tensor([[0.0, 0.0, 1.0]]) # 视线朝正 Z 方向

pts, t_vals = compute_ray_samples(O, D, num_samples=5)
print("一条射线上采集到了如下坐标点（深度）: \n", pts.squeeze(0))

### 3.4 步骤三：体渲染离散化实现
实际计算时，射线上这 $N$ 个点都会送到网络里查询得到 RGB 和 体密度 $\sigma$。紧接着，使用计算机图形学中的经典渲染方程离散化公式把它“压缩”成一个像素像素颜色：
$$\hat{C}(r) = \sum_{i=1}^N w_i c_i, \quad w_i = T_i (1 - \exp(-\sigma_i \delta_i))$$
其中 $T_i = \exp(-\sum_{j=1}^{i-1} \sigma_j \delta_j)$ 是前方所有微小容积的累积透射率，表示光线击中这一微元的概率（未被前面的物质遮挡的几率）。

In [ ]:
def raw2outputs(raw_rgbs, raw_sigmas, z_vals):
    """
    Volume rendering 体渲染核心积分计算
    """
    # 算采样点之间的距离 delta
    dists = z_vals[..., 1:] - z_vals[..., :-1]
    # 给最后一个距离补上无穷大（代表穿出了背景边界）
    dists = torch.cat([dists, torch.tensor([1e10]).expand(dists[..., :1].shape)], -1)
    
    # 透明度公式：alpha = 1 - e^(-sigma * delta)
    # ReLU保证密度总是正的
    sigma_a = F.relu(raw_sigmas) 
    alpha = 1.0 - torch.exp(-sigma_a * dists)
    
    # 透射率计算：前面所有点的 (1 - alpha) 乘积（前缀乘），为了数值稳定常加很小的极小值
    transmittance = torch.cumprod(torch.cat([torch.ones((alpha.shape[0], 1)), 1.0 - alpha + 1e-10], -1), -1)[:, :-1]
    
    # 权重 = (光透射进来的概率) * (在这个点打出来的概率)
    weights = alpha * transmittance
    
    # 对每根光线上所有点的 RGB 按照权重求和积分，获得这根射线打在屏幕上的像素颜色！
    rgb_map = torch.sum(weights.unsqueeze(-1) * raw_rgbs, -2)
    return rgb_map

# 模拟网络预测了这 5 个点的 RGB 和 $\sigma$
simulated_rgbs = torch.rand(1, 5, 3) 
simulated_sigmas = torch.tensor([[0.0, 10.0, 0.5, 0.1, 0.0]]) # 第二个点是个很硬的墙，sigma很高

pixel_color = raw2outputs(simulated_rgbs, simulated_sigmas, t_vals)
print("经过体渲染后得到的屏幕像素RGB: ", pixel_color)

---
## 4. 走向实时与显式：3D Gaussian Splatting (3DGS)

NeRF 最大的痛点：太慢！渲染一张 1080P 高清图可能需要在 MLP 内穿梭几十万次光线迭代。而 2023 年出现的 **3D Gaussian Splatting** 结合显式和隐式表达的优点，通过传统的“丢雪球式”光栅化，在保证精度的前提下跑到了数百帧（100+ FPS）。

### 4.1 基本单元：3D 高斯椭球体
3DGS 放弃了 NeRF 的全空间隐式网络建模，转而在三维空间播撒数百万个三维高斯“雪球”。每个粒子由位置 $\mu$ 和协方差 $\Sigma$ 定义：
$$G(x) = e^{-\frac{1}{2}(x-\mu)^T \Sigma^{-1} (x-\mu)}$$

为了保证训练过程中协方差矩阵的正定性（不能成反面或者扁成一条线），协方差矩阵是被分解成旋转缩放学习的：
$$\Sigma = R S S^T R^T$$
- $R(3\times3)$ 旋转矩阵，由网络直接学得四元数（Quaternion）计算而来。
- $S(3\times3)$ 缩放向量产生的对角矩阵，控制这个高斯椭球各个轴的长短。

In [ ]:
def build_covariance(scale: torch.Tensor, rot_mat: torch.Tensor):
    """
    根据给定的三维缩放因子和旋转矩阵构建三维高斯协方差矩阵 (Sigma = R * S * S^T * R^T)
    scale: (N, 3)
    rot_mat: (N, 3, 3)
    """
    N = scale.shape[0]
    # S 是对角阵
    S = torch.zeros((N, 3, 3))
    S[:, 0, 0] = scale[:, 0]
    S[:, 1, 1] = scale[:, 1]
    S[:, 2, 2] = scale[:, 2]
    
    # L = R * S
    L = torch.bmm(rot_mat, S)
    # Sigma = L * L^T = R * S * S^T * R^T
    Cov = torch.bmm(L, L.transpose(1, 2))
    
    return Cov

# 演示代码
# 想象一个粒子在 Z方向 上拉长了 10 倍，而 X, Y 半径为 1
scales = torch.tensor([[1.0, 1.0, 10.0]])
rot = torch.eye(3).unsqueeze(0) # 没旋转过
cov = build_covariance(scales, rot)

print("生成的高斯协方差矩阵:\n", cov)

### 4.2 Tiled Rasterization 加速原理 (为什么快)
之所以 3DGS 极快无比，是因为它采用了瓦片式（Tiled）架构：
1. **预处理**: 使用相机的 $E$ 和 $K$ 矩阵对 3D 高斯投影至 2D 平面上降维成 2D 椭球体（雅可比变换）。
2. **切分 Tile**: 将整张图像切分为 16x16 像素大小的区块 (tiles)。
3. **剔除与排序**: 并行地确定每个高斯“雪球”遮蔽了哪些 tile，在每一个 tile 内部只保留相交的高斯。根据它们到摄像机的**深度 (Depth) 快速进行从前到后的 Radix Sort**。
4. **渲染绘制**: 从前往后快速做 Alpha Blending 的颜色累加绘制，当透明度达到极限时直接停止该像素渲染（不再往后透视背景）。由于 GPU 天然适合在 Grid 显存结构中用这种 Tile 方法操作，因此极其高效！